<a href="https://colab.research.google.com/github/SivaSam10/SivaSam/blob/main/Healthcare_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Healthcare Q&A Chatbot with RAG
### Built using LlamaIndex + LangChain + Google Gemini API

**College Assignment Project — Beginner Friendly**

This notebook builds a **Retrieval-Augmented Generation (RAG)** chatbot that answers basic healthcare questions using a small custom knowledge base (symptoms, first aid, general wellness tips, etc.).

**Pipeline overview:**
1. Load / create a healthcare knowledge base (plain text documents).
2. Convert documents into vector embeddings using `sentence-transformers`.
3. Index the embeddings using **LlamaIndex** + **FAISS**.
4. Use **LangChain** to build the prompt template and manage the retrieval + generation chain.
5. Use **Google Gemini API** (or OpenAI as a fallback) as the LLM that generates the final answer.
6. Ask the chatbot 5 sample healthcare questions and see the responses.

> ⚠️ **Disclaimer:** This chatbot is for educational purposes only. It does **not** provide medical advice, diagnosis, or treatment. Always consult a licensed healthcare professional for real medical concerns.


## Step 1 — Install Required Libraries

We install:
- `llama-index` — for building and querying the vector index
- `llama-index-embeddings-huggingface` — to use `sentence-transformers` embeddings inside LlamaIndex
- `llama-index-llms-gemini` — to connect LlamaIndex to Google Gemini
- `langchain` / `langchain-google-genai` — for prompt templates and chain management
- `faiss-cpu` — vector similarity search engine
- `sentence-transformers` — generates semantic embeddings for our text

Run this cell first. It may take 1–2 minutes on Colab.


In [ ]:
# Install all required packages (Colab-friendly, quiet mode)
!pip install -q langchain langchain-google-genai langchain-community \
    llama-index llama-index-llms-gemini llama-index-embeddings-huggingface \
    faiss-cpu sentence-transformers google-generativeai chromadb


## Step 2 — Set Up Your Google Gemini API Key

Get a **free** Gemini API key from https://aistudio.google.com/app/apikey

We use `getpass` so the key is never printed or stored in the notebook file.

> 💡 If you'd rather use **OpenAI** instead of Gemini, see the commented alternative in the cell below.


In [ ]:
import os
from getpass import getpass

# Paste your Gemini API key when prompted (input is hidden)
os.environ['GOOGLE_API_KEY'] = getpass('Enter your Google Gemini API key: ')

# ---- OPTIONAL: OpenAI fallback instead of Gemini ----
# os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')
# USE_OPENAI = True   # set this to True and follow the alternate LLM setup in Step 5
USE_OPENAI = False


## Step 3 — Import Libraries

Importing everything we need: LlamaIndex core classes for indexing/querying, the HuggingFace embedding wrapper (uses `sentence-transformers` under the hood), the Gemini LLM wrapper, and LangChain's prompt template utilities.


In [ ]:
# --- LlamaIndex core imports (indexing & querying) ---
from llama_index.core import (
    VectorStoreIndex,
    Document,
    Settings,
    StorageContext,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore

# --- Embedding model (sentence-transformers via HuggingFace wrapper) ---
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# --- LLM: Google Gemini ---
from llama_index.llms.gemini import Gemini

# --- LangChain: prompt template management ---
from langchain.prompts import PromptTemplate

import faiss
print('All libraries imported successfully ✅')


## Step 4 — Create the Healthcare Knowledge Base

Since no dataset was provided, we create a small **custom healthcare knowledge base** as plain text documents. In a real project you could replace this with PDFs/text files loaded from Google Drive using `SimpleDirectoryReader`.

Each entry below covers a common, safe, general-knowledge health topic (symptoms, first aid basics, wellness tips) — **not** prescriptive medical advice.


In [ ]:
# A small custom healthcare knowledge base.
# Each string acts as one 'document' covering a specific topic.
healthcare_knowledge_base = [
    '''Common Cold: The common cold is a viral infection of the nose and throat.
    Symptoms include a runny or stuffy nose, sneezing, sore throat, mild cough, and
    mild fatigue. It usually resolves on its own within 7 to 10 days. Rest, drinking
    plenty of fluids, and over-the-counter remedies for symptom relief are commonly
    recommended. See a doctor if symptoms last more than 10 days or worsen.''',

    '''Seasonal Flu (Influenza): The flu is a contagious respiratory illness caused by
    influenza viruses. Symptoms include fever, chills, muscle aches, cough, congestion,
    headache, and fatigue, often appearing suddenly. Rest and hydration are important.
    Annual flu vaccination is recommended for prevention, especially for high-risk groups.''',

    '''Basic First Aid for Minor Cuts: Clean the wound gently with running water, apply
    mild antiseptic, and cover it with a sterile bandage. Change the bandage daily and
    keep the area clean and dry. Seek medical attention if the cut is deep, does not
    stop bleeding after 10 minutes of firm pressure, or shows signs of infection like
    redness, swelling, warmth, or pus.''',

    '''Managing a Mild Fever: A mild fever (up to 100.9°F / 38.3°C) in adults can often
    be managed with rest, fluids, and light clothing. Over-the-counter fever reducers
    like paracetamol may help. Seek medical care if fever exceeds 103°F (39.4°C),
    lasts more than 3 days, or is accompanied by severe headache, rash, or difficulty
    breathing.''',

    '''Healthy Sleep Habits: Adults generally need 7 to 9 hours of sleep per night for
    good health. Good sleep hygiene includes keeping a consistent sleep schedule,
    avoiding caffeine and screens before bedtime, keeping the bedroom dark and cool,
    and getting regular daytime exercise. Poor sleep is linked to weakened immunity,
    poor concentration, and increased stress.''',

    '''Balanced Diet Basics: A balanced diet includes a variety of fruits, vegetables,
    whole grains, lean proteins, and healthy fats. Adults are generally encouraged to
    eat at least 5 servings of fruits and vegetables daily, limit added sugar and
    processed foods, and stay hydrated with about 2 to 3 liters of water per day
    depending on activity level and climate.''',

    '''Understanding Blood Pressure: Normal blood pressure for adults is typically
    around 120/80 mmHg. Hypertension (high blood pressure) is generally diagnosed at
    readings of 130/80 mmHg or higher on repeated measurement. Risk factors include
    high salt intake, obesity, lack of exercise, stress, and smoking. Regular
    monitoring and a doctor's guidance are important for management.''',

    '''Staying Hydrated: Water supports digestion, circulation, temperature regulation,
    and nutrient transport. Mild dehydration symptoms include thirst, dry mouth,
    fatigue, and dark yellow urine. A common general guideline is about 8 cups
    (roughly 2 liters) of fluids per day for adults, adjusted for exercise, heat,
    and individual needs.''',

    '''When to See a Doctor for a Headache: Most headaches are mild and resolve with
    rest, hydration, and over-the-counter pain relief. Seek prompt medical attention
    for a sudden, severe 'worst-ever' headache, headache with fever and stiff neck,
    headache after a head injury, or headache with confusion, vision changes, or
    slurred speech, as these can indicate a serious condition.''',

    '''Benefits of Regular Exercise: Regular physical activity (about 150 minutes of
    moderate exercise per week for adults) helps maintain a healthy weight, strengthens
    the heart and muscles, improves mood and sleep, and reduces the risk of chronic
    diseases such as type 2 diabetes and heart disease. Always start gradually and
    consult a doctor before beginning a new intense exercise program if you have
    existing health conditions.''',
]

print(f'Knowledge base created with {len(healthcare_knowledge_base)} documents ✅')


## Step 5 — Generate Embeddings & Build the LlamaIndex Vector Index

1. Wrap each text entry as a LlamaIndex `Document`.
2. Split documents into smaller chunks (`SentenceSplitter`) — good practice for retrieval quality.
3. Use a `sentence-transformers` model (`all-MiniLM-L6-v2`) to generate embeddings for each chunk.
4. Store the embeddings in a **FAISS** vector store for fast semantic search.
5. Set Gemini as the global LLM used by LlamaIndex for answer generation.


In [ ]:
# 1. Convert each raw text entry into a LlamaIndex Document object
documents = [Document(text=text) for text in healthcare_knowledge_base]

# 2. Configure the embedding model (sentence-transformers, runs locally, free)
embed_model = HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2')

# 3. Configure the Gemini LLM used for generating the final answer
llm = Gemini(model='models/gemini-1.5-flash', api_key=os.environ['GOOGLE_API_KEY'])

# 4. Register embed_model and llm as global defaults for LlamaIndex
Settings.embed_model = embed_model
Settings.llm = llm
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=20)

# 5. Create a FAISS vector store sized to match the embedding dimension (384 for MiniLM-L6-v2)
faiss_dim = 384
faiss_index = faiss.IndexFlatL2(faiss_dim)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 6. Build the vector index: this chunks the documents, embeds each chunk,
#    and stores the vectors in FAISS for semantic similarity search
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

print('Vector index built successfully ✅')


## Step 6 — Build the RAG Prompt Template with LangChain

LangChain's `PromptTemplate` lets us clearly define the instructions given to the LLM: answer **only** using the retrieved context, stay factual, and add a safety disclaimer. This template is inserted into the LlamaIndex query engine as a custom QA prompt, which is how LangChain and LlamaIndex work together in this pipeline.


In [ ]:
# Define a custom prompt template using LangChain's PromptTemplate.
# {context_str} = retrieved chunks from the knowledge base (filled in automatically)
# {query_str}    = the user's question
healthcare_prompt_template = PromptTemplate(
    input_variables=['context_str', 'query_str'],
    template=(
        'You are a helpful healthcare information assistant.\n'
        'Answer the question using ONLY the context below. If the answer is not '
        'in the context, say you do not have enough information.\n'
        'Keep the answer concise (2-4 sentences) and end with a brief reminder '
        'to consult a doctor for personal medical advice.\n\n'
        'Context:\n{context_str}\n\n'
        'Question: {query_str}\n'
        'Answer:'
    ),
)

# LlamaIndex needs its own PromptTemplate object (built from the same text)
# to plug into the query engine below.
from llama_index.core import PromptTemplate as LlamaPromptTemplate

llama_qa_prompt = LlamaPromptTemplate(
    healthcare_prompt_template.template
    .replace('{context_str}', '{context_str}')
    .replace('{query_str}', '{query_str}')
)

print('Prompt template ready ✅')
print(healthcare_prompt_template.template)


## Step 7 — Build the Retrieval-Augmented Generation (RAG) Query Engine

This is where retrieval and generation come together:
1. `as_query_engine()` turns our vector index into a searchable engine.
2. `similarity_top_k=3` retrieves the 3 most semantically relevant chunks for each question.
3. We inject our custom LangChain-style prompt so answers are grounded in the retrieved context.
4. The Gemini LLM generates the final natural-language answer from the retrieved context.


In [ ]:
# Build the RAG query engine from our vector index
query_engine = index.as_query_engine(
    similarity_top_k=3,                 # retrieve top-3 most relevant chunks
    text_qa_template=llama_qa_prompt,    # use our custom healthcare prompt
)

def ask_healthcare_bot(question: str) -> str:
    '''
    Sends a user question through the RAG pipeline:
    1. Embed the question
    2. Retrieve the most relevant chunks from FAISS
    3. Feed context + question to Gemini via the custom prompt
    4. Return the generated answer
    '''
    response = query_engine.query(question)
    return str(response)

print('RAG query engine ready ✅')


## Step 8 — Test the Chatbot with 5 Sample Questions

Let's ask the chatbot 5 different healthcare questions and print the responses.


In [ ]:
sample_questions = [
    'What are the symptoms of a common cold?',
    'How should I treat a minor cut at home?',
    'How much sleep do adults need each night?',
    'What is considered a normal blood pressure reading?',
    'When should I see a doctor for a headache?',
]

for i, question in enumerate(sample_questions, start=1):
    answer = ask_healthcare_bot(question)
    print(f'Q{i}: {question}')
    print(f'A{i}: {answer}')
    print('-' * 80)


## Expected Output (example)

```
Q1: What are the symptoms of a common cold?
A1: Common cold symptoms include a runny or stuffy nose, sneezing, sore throat,
    mild cough, and mild fatigue, usually resolving within 7-10 days. Rest and
    fluids are commonly recommended. Please consult a doctor for personal advice.
--------------------------------------------------------------------------------
Q2: How should I treat a minor cut at home?
A2: Clean the wound with running water, apply mild antiseptic, and cover it with
    a sterile bandage, changing it daily. Seek medical help if bleeding doesn't
    stop or signs of infection appear. Please consult a doctor for personal advice.
--------------------------------------------------------------------------------
Q3: How much sleep do adults need each night?
A3: Adults generally need 7 to 9 hours of sleep per night, supported by a
    consistent schedule and good sleep hygiene. Please consult a doctor for
    personal advice.
--------------------------------------------------------------------------------
Q4: What is considered a normal blood pressure reading?
A4: Normal blood pressure is typically around 120/80 mmHg, while readings of
    130/80 mmHg or higher are generally considered hypertension. Please consult
    a doctor for personal advice.
--------------------------------------------------------------------------------
Q5: When should I see a doctor for a headache?
A5: See a doctor promptly for a sudden severe headache, one with fever and stiff
    neck, after a head injury, or with confusion or vision changes. Please
    consult a doctor for personal advice.
--------------------------------------------------------------------------------
```

> Actual wording will vary slightly each run since it's generated live by Gemini, but answers should stay grounded in the knowledge base above.


## Step 9 (Optional) — Interactive Chat Loop

Run this cell to chat with the bot interactively in Colab. Type `exit` to stop.


In [3]:
# Optional interactive loop — run this cell and type your own questions
while True:
    user_input = input('You: ')
    if user_input.strip().lower() in ['exit', 'quit']:
        print('Chatbot: Take care! 👋')
        break
    reply = ask_healthcare_bot(user_input)
    print(f'Chatbot: {reply}\n')


You: quit
Chatbot: Take care! 👋


## Notes & Possible Extensions

- **Swap in real data:** Replace `healthcare_knowledge_base` with `SimpleDirectoryReader('your_folder/').load_data()` to index PDFs or `.txt` files (e.g. from WHO/CDC public fact sheets).
- **Use ChromaDB instead of FAISS:** Replace the `FaissVectorStore` block with `llama_index.vector_stores.chroma.ChromaVectorStore` if you prefer a persistent, disk-backed vector database.
- **Switch to OpenAI:** Set `USE_OPENAI = True` in Step 2, then replace the Gemini LLM in Step 5 with `from llama_index.llms.openai import OpenAI` and `llm = OpenAI(model='gpt-4o-mini')`.
- **Add a simple UI:** Wrap `ask_healthcare_bot()` with `gradio` (`!pip install gradio`) for a shareable web chat interface — a nice addition for a project demo.
- **Safety:** This is a general-knowledge demo, not a diagnostic tool. Always include a disclaimer in any real deployment.
